In [2]:
import pandas as pd
from pathlib import Path
PROJECT_ROOT = next(
    p for p in Path.cwd().parents if p.name == "LaboroTomato"
)

In [3]:
def main():
    RUN_DIR = PROJECT_ROOT / "runs_cpu_local" / "baseline_mobilenetv2" / "20251228-003415"
    pred_csv = RUN_DIR / "predictions.csv"
    assert pred_csv.exists(), f"Missing: {pred_csv}"

    df = pd.read_csv(pred_csv)

    # Ensure expected columns exist
    required = {"path", "true_label", "pred_label", "confidence", "correct"}
    missing = required - set(df.columns)
    assert not missing, f"predictions CSV missing columns: {missing}"

    # 10 correct per class: pick highest-confidence correct (clean exemplars for figures)
    correct_df = df[df["correct"] == 1].copy()
    correct_picks = (
        correct_df.sort_values("confidence", ascending=False)
        .groupby("true_label", group_keys=False)
        .head(10)
    )

    # Wrong picks: focus on green <-> half_ripened confusion first
    wrong_df = df[df["correct"] == 0].copy()

    # prioritize: true green predicted half, true half predicted green
    gh = wrong_df[
        ((wrong_df["true_label"] == "green") & (wrong_df["pred_label"] == "half_ripened")) |
        ((wrong_df["true_label"] == "half_ripened")
         & (wrong_df["pred_label"] == "green"))
    ].sort_values("confidence", ascending=False)

    # if insufficient, add other mistakes by confidence
    remaining_needed = max(0, 10 - len(gh))
    other_wrong = wrong_df.drop(gh.index, errors="ignore").sort_values(
        "confidence", ascending=False)

    wrong_picks = pd.concat(
        [gh.head(10), other_wrong.head(remaining_needed)], ignore_index=True)

    # Combine and add a 'subset' tag
    correct_picks = correct_picks.copy()
    correct_picks["subset"] = "correct"

    wrong_picks = wrong_picks.copy()
    wrong_picks["subset"] = "wrong"

    samples = pd.concat([correct_picks, wrong_picks], ignore_index=True)

    # Add stable id for filenames
    samples = samples.reset_index(drop=True)
    samples["sample_id"] = samples.index.astype(int)

    # Save
    out_csv = RUN_DIR / "samples.csv"
    samples.to_csv(out_csv, index=False)

    print(f"Saved: {out_csv}")
    print("Counts:")
    print(samples["subset"].value_counts())
    print("\nCorrect per class:")
    print(samples[samples["subset"] == "correct"]["true_label"].value_counts())
    print("\nWrong type breakdown:")
    print(samples[samples["subset"] == "wrong"]
          [["true_label", "pred_label"]].value_counts().head(10))

In [4]:
if __name__ == "__main__":
    main()

Saved: c:\Users\User\Desktop\LaboroTomato\runs_cpu_local\baseline_mobilenetv2\20251228-003415\samples.csv
Counts:
subset
correct    30
wrong      10
Name: count, dtype: int64

Correct per class:
true_label
green            10
fully_ripened    10
half_ripened     10
Name: count, dtype: int64

Wrong type breakdown:
true_label    pred_label
half_ripened  green         10
Name: count, dtype: int64


In [1]:
!jupyter nbconvert --to html 05a_select_samples.ipynb

[NbConvertApp] Converting notebook 05a_select_samples.ipynb to html
[NbConvertApp] Writing 284580 bytes to 05a_select_samples.html
